# SSO Signup Optimization V2 — Power Analysis (First Fix Conversion)

**Experiment:** SSO Signup Optimization V2 (Control vs. Treatment) · **Primary metric:** First Fix conversion · **Randomization unit:** `visitor_id`

## Background

A prior iteration of this signup experience test compared a broader SSO-first treatment (surfacing Google, Email, Apple, and Facebook signup options prominently) against the current signup page. First Fix conversion was tracked only as a secondary signal in that test, but the SSO-first treatment showed a directional +3.8% relative lift on it — a result that did not reach conventional statistical significance, though there was 97% confidence the treatment effect was positive. Because First Fix conversion was not the metric that test was sized around, that result was directionally encouraging but inconclusive.

This iteration narrows the treatment to **Google + Email only** on desktop and mobile web (removing Apple, which has a materially lower authentication success rate than the other options, and Facebook, which is being deprecated), and **First Fix conversion is the primary metric this test is powered around**. The question this experiment is designed to answer is whether the directional downstream conversion signal is real and repeatable once the test is properly powered to detect it.

## Population & metric definition

- **Population:** visitors reaching the signup page for the first time each month, on desktop or mobile web (`curated.product_tracking_events`, restricted to `platform = 'web'`), excluding visitors who had already converted before that particular visit. The iOS app is out of scope for this test and is naturally excluded, since native app signup doesn't route through this web page.
- **Primary metric — First Fix conversion:** whether the visitor requested a First Fix within 7 days of reaching the signup page, approximated via `curated.user_session_conversion_metrics.request_7d_flag` aggregated per visitor. There is no raw "request timestamp" column on this table, so the precomputed 7-day flag is taken at face value rather than re-derived against the exact visit timestamp — a known simplification.

## Design parameters

- **Two-arm test:** Control (current signup experience) vs. Treatment (simplified SSO-first: Google presented first, Email as the clear alternative, Apple and Facebook removed).
- **Traffic split:** 50/50 — the standard even allocation for a two-arm test absent a specific reason to skew it.
- **Significance level:** alpha = 0.05, two-sided. A two-sided test is used even though the success criterion is a positive lift, since it also protects against concluding "no effect" when the true effect is an unexpected negative one. Only one treatment-vs-control comparison is planned, so no multiple-comparison adjustment is needed.
- **Power:** 80%, the conventional target for treating a null result as informative rather than inconclusive.
- **Minimum detectable effect:** relative lifts of 2%, 3%, 5%, and 10% on the First Fix conversion baseline. The 3% relative lift is the primary scenario for the sample-size and duration figures in this analysis; the other checkpoints show sensitivity to the assumed effect size.
- Sample sizes are computed with `n_total_statsmodels` — a two-proportion z-test sample-size/power solver defined in `power.py` (same directory), which inverts `statsmodels.stats.proportion.power_proportions_2indep` via root-finding to solve for the per-arm sample size that hits a target power.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters (used by the warehouse query in Step 1)
SIGNUP_URL_PATTERN = 'https://www.stitchfix.com/signup%'
START_DATE         = '2026-01-01'
CONV_WINDOW_DAYS   = 7  # "requested a First Fix up to 7 days" after the signup-page visit

# Design parameters (two-arm test, one planned comparison)
ALPHA = 0.05
POWER = 0.80
TWO_SIDED = True
SPLIT_RATIO = 0.5
N_ARMS = 2
MDE_GRID = [0.02, 0.03, 0.05, 0.10]  # relative lifts on First Fix conversion sized below
HEADLINE_MDE = 0.03

## Step 1 — Warehouse baseline: First Fix conversion

One row per visitor per month, anchored to the first web signup-page visit that month, excluding visitors who had already converted before that visit. Aggregated monthly to get daily signup-page reach and the First Fix conversion rate.

In [2]:
first_fix_query = f"""--sql
WITH signup_page_visits AS (
    -- One row per visitor per month, anchored to their first signup-page visit that month.
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '{START_DATE}'
      AND url LIKE '{SIGNUP_URL_PATTERN}'
      AND platform = 'web'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    -- One row per visitor: whether they ever signed up, and whether any of their sessions
    -- carries the table's own "requested a First Fix within 7 days" flag.
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts,
        MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '{START_DATE}'
    GROUP BY visitor_id
),
classified AS (
    -- Only visitors who had not already converted before this visit count as "new"
    -- for this month's cohort.
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
    FROM signup_page_visits v
    LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
),
month_days AS (
    -- Distinct calendar days with a signup-page visit, used to average daily volume.
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS first_fix_requests,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS first_fix_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

first_fix_df = query(first_fix_query)
first_fix_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,first_fix_requests,first_fix_conv_rate,signup_page_visitors_per_day
0,2026-08-01 00:00:00.000,21,224298,23555,0.105017,10680
1,2026-07-01 00:00:00.000,31,295522,30197,0.102182,9532
2,2026-06-01 00:00:00.000,30,225103,22707,0.100874,7503
3,2026-05-01 00:00:00.000,31,269030,30226,0.112352,8678
4,2026-04-01 00:00:00.000,30,288653,33199,0.115014,9621
5,2026-03-01 00:00:00.000,31,368601,42812,0.116147,11890
6,2026-02-01 00:00:00.000,28,335632,36133,0.107657,11986
7,2026-01-01 00:00:00.000,31,358778,38036,0.106015,11573


**Reference month.** The most recent fully-observed calendar month is used as the baseline, since it's the freshest complete read on current funnel behavior (the current month is still in progress and would understate volume and bias the conversion rate).

In [3]:
REFERENCE_MONTH = first_fix_df['month'].astype(str).str.slice(0, 7).iloc[1]  # most recent complete month (index 0 is the in-progress current month)

ff_row = first_fix_df[first_fix_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_FF = float(ff_row['first_fix_conv_rate'][0])
DAILY_ELIGIBLE_VISITORS = float(ff_row['signup_page_visitors_per_day'][0])

print(f"Reference month: {REFERENCE_MONTH}")
print(f"BASELINE_FF (First Fix conversion) = {BASELINE_FF:.4f}")
print(f"DAILY_ELIGIBLE_VISITORS            = {DAILY_ELIGIBLE_VISITORS:,.0f}")

Reference month: 2026-07
BASELINE_FF (First Fix conversion) = 0.1022
DAILY_ELIGIBLE_VISITORS            = 9,532


## Step 2 — Sample size and duration required

`n_total_statsmodels` returns the **per-arm** sample size; with a 50/50 split and two arms, `n_total = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE_VISITORS / 2` per day.

In [4]:
def size_table(rel_grid, baseline, daily, label):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT_RATIO],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.2%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total'] = df['n_per_arm'] * N_ARMS
    if daily:
        df['days_required'] = np.ceil(df['n_total'] / daily).astype(int)
        df['weeks_required'] = (df['days_required'] / 7).round(1)
        cols = ['rel_effect', 'p_treatment', 'n_per_arm', 'n_total', 'days_required', 'weeks_required']
    else:
        cols = ['rel_effect', 'p_treatment', 'n_per_arm', 'n_total']

    sided = 'two-sided' if TWO_SIDED else 'one-sided'
    print(f"--- {label} (baseline={baseline:.2%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, split={SPLIT_RATIO:.0%}/{1-SPLIT_RATIO:.0%}) ---")

    return df[cols]

ff_size_df = size_table(MDE_GRID, BASELINE_FF, DAILY_ELIGIBLE_VISITORS, 'First Fix conversion')
ff_size_df

--- First Fix conversion (baseline=10.22%, alpha=0.05, power=80%, two-sided, split=50%/50%) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2.00%,0.104226,347870,695740,73,10.4
1,+3.00%,0.105247,155285,310570,33,4.7
2,+5.00%,0.107291,56389,112778,12,1.7
3,+10.00%,0.1124,14399,28798,4,0.6


**Reading this:** at the headline **+3% relative lift**, 80% power requires **~155.3k visitors per arm (~310.6k total)** — about **4.7 weeks** of current signup-page traffic. Because First Fix's baseline rate (~10%) is much lower than a shallower funnel metric, the same *relative* lift is a smaller *absolute* lift here, which takes substantially more data to distinguish from noise — a 2% relative lift would take over 10 weeks, while a 10% relative lift would take under a week.

## Step 3 — Summary

Required sample size and duration for a 3% relative lift on First Fix conversion, the primary scenario for this design.

In [5]:
HEADLINE_ROW = ff_size_df[ff_size_df['rel_effect'] == f"{HEADLINE_MDE:+.2%}"].iloc[0]
n_per_arm_headline = int(HEADLINE_ROW['n_per_arm'])
duration_days = int(HEADLINE_ROW['days_required'])
duration_weeks = round(duration_days / 7, 1)

summary = {
    'Metric Used':                    'Visitor First Fix Request Flag Up to 7 Days',
    'Metrics Baseline Value':         f"{BASELINE_FF:.2%} (warehouse baseline, {REFERENCE_MONTH}, new visitors reaching the web signup page)",
    'Minimum Detectable Effect':      f"+{HEADLINE_MDE:.0%} relative",
    'One/Two-Sided Test':             'Two-sided',
    'Significance Level':             f"{ALPHA}",
    'Statistical Power':              f"{POWER:.0%}",
    'Variant Split % (T vs. C)':      f"{SPLIT_RATIO:.0%} / {1 - SPLIT_RATIO:.0%}",
    'Minimum Samples by Variant':     f"{n_per_arm_headline:,}",
    'Minimum Samples total':          f"{n_per_arm_headline * N_ARMS:,}",
    'Shortest Duration Required':     f"{duration_days} days ({duration_weeks} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Visitor First Fix Request Flag Up to 7 Days
Metrics Baseline Value,"10.22% (warehouse baseline, 2026-07, new visitors reaching the web signup page)"
Minimum Detectable Effect,+3% relative
One/Two-Sided Test,Two-sided
Significance Level,0.05
Statistical Power,80%
Variant Split % (T vs. C),50% / 50%
Minimum Samples by Variant,"155,285"
Minimum Samples total,"310,570"
Shortest Duration Required,33 days (4.7 weeks)


## Bottom line

- Detecting a 3% relative lift in First Fix conversion at 80% power requires about **4.7 weeks** of the current signup-page traffic (~155.3k visitors per arm, ~310.6k total).
- Required sample size is highly sensitive to the assumed effect size given First Fix's relatively low baseline rate (~10%): a 2% relative lift would need over 10 weeks, while a 10% relative lift would need under a week. A true effect smaller than 3% would be underpowered at the sample sizes above.